# SageMaker Pipeline and Model Registry



# Thomas Geraci

## 1. AWS and SageMaker Setup

In [1]:
import os
import json
import time
import boto3
import logging
import warnings

os.environ.setdefault("SAGEMAKER_SUPPRESS_V2_WARNING", "1")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

logging.getLogger("sagemaker.workflow.utilities").setLevel(logging.ERROR)
logging.getLogger("sagemaker.estimator").setLevel(logging.ERROR)
logging.getLogger("sagemaker.telemetry.telemetry_logging").setLevel(logging.ERROR)

warnings.filterwarnings(
    "ignore",
    message=r".*PipelineSession.*No Wait.*No Logs.*No Job being started.*",
    category=UserWarning
)

logging.getLogger("sagemaker.workflow._utils").setLevel(logging.ERROR)
logging.getLogger("sagemaker.workflow.pipeline_context").setLevel(logging.ERROR)

import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession

role_arn = os.environ.get("SAGEMAKER_ROLE_ARN")

if not role_arn:
    try:
        from sagemaker import get_execution_role
        role_arn = get_execution_role()
    except Exception as exc:
        raise RuntimeError(
            "Could not discover the SageMaker execution role. "
            "Run this notebook in SageMaker or set SAGEMAKER_ROLE_ARN."
        ) from exc

boto_session = boto3.Session()
region = boto_session.region_name or "us-east-1"
account_id = boto_session.client("sts").get_caller_identity()["Account"]

pipeline_session = PipelineSession(
    boto_session=boto_session,
    sagemaker_client=boto_session.client("sagemaker")
)

sm_client = boto_session.client("sagemaker", region_name=region)
s3_client = boto_session.client("s3", region_name=region)

print("AWS Region:", region)
print("AWS Account:", account_id)
print("SageMaker Role:", role_arn)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
AWS Region: us-east-1
AWS Account: 443910721300
SageMaker Role: arn:aws:iam::443910721300:role/LabRole


## 2. Shared Project Configuration and Input Validation

In [2]:
from sagemaker.workflow.parameters import (
    ParameterFloat,
    ParameterString
)

model_bucket = f"sagemaker-aai540-modeling-{account_id}"
model_prefix = "flight-delay-xgboost"

train_s3_path = (
    f"s3://{model_bucket}/{model_prefix}/"
    "data-uncompressed/train/"
)
validation_s3_path = (
    f"s3://{model_bucket}/{model_prefix}/"
    "data-uncompressed/validation/"
)
test_s3_path = (
    f"s3://{model_bucket}/{model_prefix}/"
    "data-uncompressed/test/"
)

pipeline_output_path = (
    f"s3://{model_bucket}/{model_prefix}/pipeline/"
)

target_column = "delayed"

model_features = [
    "month",
    "day_of_month",
    "day_of_week",
    "is_weekend",
    "scheduled_departure_hour",
    "scheduled_arrival_hour",
    "scheduled_duration",
    "distance"
]

# Pipeline parameters make the workflow reusable.
training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.large"
)

transform_instance_type = ParameterString(
    name="TransformInstanceType",
    default_value="ml.m5.large"
)

minimum_delayed_recall = ParameterFloat(
    name="MinimumDelayedRecall",
    default_value=0.20
)

print("Training data:", train_s3_path)
print("Validation data:", validation_s3_path)
print("Test data:", test_s3_path)
print("Pipeline outputs:", pipeline_output_path)
print("Target:", target_column)
print("Feature count:", len(model_features))

# Confirm required S3 inputs exist before defining the pipeline.
def list_prefix(uri):
    bucket, prefix = uri.replace("s3://", "").split("/", 1)
    response = s3_client.list_objects_v2(
        Bucket=bucket,
        Prefix=prefix,
        MaxKeys=5
    )
    return [item["Key"] for item in response.get("Contents", [])]

for label, uri in {
    "train": train_s3_path,
    "validation": validation_s3_path,
    "test": test_s3_path
}.items():
    files = list_prefix(uri)
    if not files:
        raise RuntimeError(
            f"No files found for {label} at {uri}. "
            "Run Model_Development.ipynb first."
        )
    print(f"{label}: FOUND ({len(files)}+ object(s))")

Training data: s3://sagemaker-aai540-modeling-443910721300/flight-delay-xgboost/data-uncompressed/train/
Validation data: s3://sagemaker-aai540-modeling-443910721300/flight-delay-xgboost/data-uncompressed/validation/
Test data: s3://sagemaker-aai540-modeling-443910721300/flight-delay-xgboost/data-uncompressed/test/
Pipeline outputs: s3://sagemaker-aai540-modeling-443910721300/flight-delay-xgboost/pipeline/
Target: delayed
Feature count: 8
train: FOUND (1+ object(s))
validation: FOUND (1+ object(s))
test: FOUND (1+ object(s))


## 3. XGBoost Estimator and Training Step

In [3]:
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

xgboost_image_uri = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1"
)

negative_records = 2214442
positive_records = 571908
scale_pos_weight = round(
    negative_records / positive_records,
    2
)

hyperparameters = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "max_depth": "5",
    "eta": "0.1",
    "min_child_weight": "1",
    "subsample": "0.8",
    "colsample_bytree": "0.8",
    "scale_pos_weight": str(scale_pos_weight),
    "num_round": "200",
    "early_stopping_rounds": "20"
}

xgb_estimator = Estimator(
    image_uri=xgboost_image_uri,
    role=role_arn,
    instance_count=1,
    instance_type=training_instance_type,
    output_path=f"{pipeline_output_path}model-output/",
    sagemaker_session=pipeline_session
)

xgb_estimator.set_hyperparameters(**hyperparameters)

step_train = TrainingStep(
    name="TrainFlightDelayXGBoost",
    estimator=xgb_estimator,
    inputs={
        "train": TrainingInput(
            s3_data=train_s3_path,
            content_type="text/csv"
        ),
        "validation": TrainingInput(
            s3_data=validation_s3_path,
            content_type="text/csv"
        )
    }
)

print("Training step created:", step_train.name)
print("XGBoost image:", xgboost_image_uri)
print("Class weight:", scale_pos_weight)

Training step created: TrainFlightDelayXGBoost
XGBoost image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
Class weight: 3.87


# Daniel Sims

## 4. Create Model and Batch Transform Step

In [4]:
from sagemaker.model import Model
from sagemaker.workflow.model_step import ModelStep
from sagemaker.transformer import Transformer
from sagemaker.workflow.steps import TransformStep

pipeline_model = Model(
    image_uri=xgboost_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role_arn,
    sagemaker_session=pipeline_session
)

create_model_args = pipeline_model.create(
    instance_type=transform_instance_type
)

step_create_model = ModelStep(
    name="CreateFlightDelayModel",
    step_args=create_model_args
)

transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_count=1,
    instance_type=transform_instance_type,
    output_path=f"{pipeline_output_path}predictions/",
    assemble_with="Line",
    accept="text/csv",
    sagemaker_session=pipeline_session
)

transform_args = transformer.transform(
    data=test_s3_path,
    content_type="text/csv",
    split_type="Line",
    input_filter="$[1:]",
    join_source="Input",
    output_filter="$[0,-1]"
)

step_transform = TransformStep(
    name="BatchInferenceTestData",
    step_args=transform_args
)

print("Model creation step:", step_create_model.name)
print("Batch Transform step:", step_transform.name)

Model creation step: CreateFlightDelayModel
Batch Transform step: BatchInferenceTestData


## 5. Automated Evaluation Processing Step

In [5]:
evaluation_script = r'''
import csv
import json
from pathlib import Path

INPUT_DIR = Path("/opt/ml/processing/input/predictions")
OUTPUT_DIR = Path("/opt/ml/processing/evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(INPUT_DIR.rglob("*.out"))

if not files:
    raise RuntimeError("No Batch Transform .out files were found.")

tp = tn = fp = fn = 0
records = 0
threshold = 0.5

for file_path in files:
    with file_path.open("r", encoding="utf-8") as stream:
        reader = csv.reader(stream)
        for row in reader:
            if len(row) < 2:
                continue

            actual = int(float(row[0]))
            probability = float(row[-1])
            predicted = int(probability >= threshold)

            records += 1

            if actual == 1 and predicted == 1:
                tp += 1
            elif actual == 0 and predicted == 0:
                tn += 1
            elif actual == 0 and predicted == 1:
                fp += 1
            elif actual == 1 and predicted == 0:
                fn += 1

if records == 0:
    raise RuntimeError("No prediction rows were available for evaluation.")

accuracy = (tp + tn) / records
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else 0.0
)

report = {
    "classification_metrics": {
        "accuracy": {"value": accuracy},
        "delayed_precision": {"value": precision},
        "delayed_recall": {"value": recall},
        "delayed_f1": {"value": f1}
    },
    "confusion_matrix": {
        "true_positive": tp,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn
    },
    "records": records,
    "threshold": threshold
}

output_file = OUTPUT_DIR / "evaluation.json"

with output_file.open("w", encoding="utf-8") as stream:
    json.dump(report, stream, indent=2)

print(json.dumps(report, indent=2))
'''

script_path = "pipeline_evaluate.py"

with open(script_path, "w", encoding="utf-8") as file:
    file.write(evaluation_script)

print("Created:", script_path)

Created: pipeline_evaluate.py


In [6]:
from sagemaker.processing import (
    ProcessingInput,
    ProcessingOutput,
    ScriptProcessor
)
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.steps import ProcessingStep

evaluation_processor = ScriptProcessor(
    image_uri=xgboost_image_uri,
    command=["python3"],
    role=role_arn,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=pipeline_session
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

evaluation_args = evaluation_processor.run(
    code=script_path,
    inputs=[
        ProcessingInput(
            source=step_transform.properties.TransformOutput.S3OutputPath,
            destination="/opt/ml/processing/input/predictions"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=f"{pipeline_output_path}evaluation/"
        )
    ]
)

step_evaluate = ProcessingStep(
    name="EvaluateFlightDelayModel",
    step_args=evaluation_args,
    property_files=[evaluation_report]
)

print("Evaluation step created:", step_evaluate.name)

Evaluation step created: EvaluateFlightDelayModel


## 6. Model-Quality Condition

In [7]:
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet

delayed_recall_value = JsonGet(
    step_name=step_evaluate.name,
    property_file=evaluation_report,
    json_path="classification_metrics.delayed_recall.value"
)

recall_condition = ConditionGreaterThanOrEqualTo(
    left=delayed_recall_value,
    right=minimum_delayed_recall
)

print(
    "Quality gate created: delayed recall must be >=",
    minimum_delayed_recall.default_value
)

Quality gate created: delayed recall must be >= 0.2


# Denis Mulabegovic

## 7. Model Registry Configuration

In [8]:
from sagemaker.model_metrics import (
    MetricsSource,
    ModelMetrics
)
from sagemaker.workflow.functions import Join
from sagemaker.workflow.step_collections import RegisterModel

model_package_group_name = "FlightDelayPrediction"

evaluation_s3_uri = Join(
    on="/",
    values=[
        step_evaluate.arguments[
            "ProcessingOutputConfig"
        ]["Outputs"][0]["S3Output"]["S3Uri"],
        "evaluation.json"
    ]
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=evaluation_s3_uri,
        content_type="application/json"
    )
)

step_register = RegisterModel(
    name="RegisterFlightDelayModel",
    estimator=xgb_estimator,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=[
        "ml.m5.large",
        "ml.m5.xlarge"
    ],
    transform_instances=[
        "ml.m5.large",
        "ml.m5.xlarge"
    ],
    model_package_group_name=model_package_group_name,
    approval_status="PendingManualApproval",
    model_metrics=model_metrics
)

step_quality_gate = ConditionStep(
    name="CheckDelayedRecall",
    conditions=[recall_condition],
    if_steps=[step_register],
    else_steps=[]
)

print("Model package group:", model_package_group_name)
print("Approval status: PendingManualApproval")
print("Registration will occur only if the quality gate passes.")

Model package group: FlightDelayPrediction
Approval status: PendingManualApproval
Registration will occur only if the quality gate passes.


## 8. Build and Execute the SageMaker Pipeline

In [9]:
from sagemaker.workflow.pipeline import Pipeline

pipeline_name = "FlightDelayPredictionPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        training_instance_type,
        transform_instance_type,
        minimum_delayed_recall
    ],
    steps=[
        step_train,
        step_create_model,
        step_transform,
        step_evaluate,
        step_quality_gate
    ],
    sagemaker_session=pipeline_session
)

pipeline_definition = pipeline.definition()

print("Pipeline name:", pipeline_name)
print("Pipeline definition created successfully.")
print("Definition length:", len(pipeline_definition))

Pipeline name: FlightDelayPredictionPipeline
Pipeline definition created successfully.
Definition length: 6145


In [10]:
# Create or update the pipeline definition in SageMaker.
upsert_response = pipeline.upsert(
    role_arn=role_arn,
    description=(
        "AAI-540 Flight Delay Prediction training, evaluation, "
        "quality gate, and model registration pipeline."
    )
)

print("Pipeline upsert complete.")
print("Pipeline ARN:", upsert_response.get("PipelineArn"))

# Start a normal execution using the default quality threshold.
execution = pipeline.start()

print("Pipeline execution started.")
print("Execution ARN:", execution.arn)

Pipeline upsert complete.
Pipeline ARN: arn:aws:sagemaker:us-east-1:443910721300:pipeline/FlightDelayPredictionPipeline
Pipeline execution started.
Execution ARN: arn:aws:sagemaker:us-east-1:443910721300:pipeline/FlightDelayPredictionPipeline/execution/y59ydbzvwsr4


## 9. Verify Pipeline Execution and Registered Model

In [11]:
execution.wait()

execution_description = execution.describe()

print(
    "Pipeline execution status:",
    execution_description["PipelineExecutionStatus"]
)

print("\nStep results:")
step_results = execution.list_steps()

for step in reversed(step_results):
    print(
        f"- {step['StepName']}: "
        f"{step['StepStatus']}"
    )

if execution_description["PipelineExecutionStatus"] != "Succeeded":
    failure_reason = execution_description.get(
        "FailureReason",
        "No failure reason returned."
    )
    raise RuntimeError(
        "Pipeline did not succeed: " + failure_reason
    )

print("\nPipeline completed successfully.")

Pipeline execution status: Succeeded

Step results:
- TrainFlightDelayXGBoost: Succeeded
- CreateFlightDelayModel-CreateModel: Succeeded
- BatchInferenceTestData: Succeeded
- EvaluateFlightDelayModel: Succeeded
- CheckDelayedRecall: Succeeded
- RegisterFlightDelayModel-RegisterModel: Succeeded

Pipeline completed successfully.


In [12]:
# Verify Model Registry contents.
try:
    group = sm_client.describe_model_package_group(
        ModelPackageGroupName=model_package_group_name
    )
    print(
        "Model Package Group:",
        group["ModelPackageGroupName"]
    )
except sm_client.exceptions.ResourceNotFound:
    raise RuntimeError(
        "Model Package Group was not created. "
        "Check whether the quality condition passed."
    )

packages = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=10
)["ModelPackageSummaryList"]

if not packages:
    raise RuntimeError(
        "No registered model versions were found. "
        "Check the quality-gate result."
    )

print("\nRegistered model versions:")

for package in packages:
    print(
        "Version:",
        package["ModelPackageVersion"],
        "| Status:",
        package["ModelPackageStatus"],
        "| Approval:",
        package["ModelApprovalStatus"]
    )

latest_package_arn = packages[0]["ModelPackageArn"]

latest_package = sm_client.describe_model_package(
    ModelPackageName=latest_package_arn
)

print("\nLatest registered model:")
print("ARN:", latest_package_arn)
print(
    "Approval:",
    latest_package["ModelApprovalStatus"]
)
print(
    "Created:",
    latest_package["CreationTime"]
)

Model Package Group: FlightDelayPrediction

Registered model versions:
Version: 4 | Status: Completed | Approval: PendingManualApproval
Version: 3 | Status: Completed | Approval: PendingManualApproval
Version: 2 | Status: Completed | Approval: PendingManualApproval
Version: 1 | Status: Completed | Approval: PendingManualApproval

Latest registered model:
ARN: arn:aws:sagemaker:us-east-1:443910721300:model-package/FlightDelayPrediction/4
Approval: PendingManualApproval
Created: 2026-10-06 23:05:30.811000+00:00
